- Imports

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from scipy.ndimage import label, binary_dilation
from src import src, misc 
import math

print(tf.config.list_physical_devices('GPU')) 

## imaging forward model

In [ ]:
import os
import numpy as np
import tensorflow as tf
from scipy.ndimage import label, binary_dilation

# ============================================================
# Shared raw-data loader for the same 10 files
# ============================================================
def load_raw_by_select(raw_select, data_dir="data", aug_prefix="aug"):
    """
    Mapping:
      1 -> y_exp_test_4.npy
      2 -> aug_1.npy
      3 -> aug_2.npy
      ...
      10 -> aug_9.npy

    Returns:
      y_tf  : (1,K,T,R) tf.complex64
      y_np  : (K,T,R) np.complex64
      path  : file path
    """
    raw_select = int(raw_select)

    if raw_select == 1:
        path = os.path.join(data_dir, "y_exp_test_4.npy")
    elif 2 <= raw_select <= 10:
        path = os.path.join(data_dir, f"{aug_prefix}_{raw_select - 1}.npy")
    else:
        raise ValueError("raw_select must be between 1 and 10")

    if not os.path.isfile(path):
        raise FileNotFoundError(f"Raw file not found: {path}")

    y_np = np.load(path).astype(np.complex64)              # (K,T,R)
    y_tf = tf.constant(y_np[None, ...], dtype=tf.complex64)  # (1,K,T,R)
    return y_tf, y_np, path


# ============================================================
# Model pipeline
# ============================================================
def get_model_pipeline(model_name, X, Y, Z):
    if model_name == 'Deep2S':
        dnn = src.get_UNet3D(input_shape=(X, Y, Z, 1))
        dnn.load_weights("models_weights/experimental/Deep2S/model_Nf15_SNR30_exp.h5")

        def forward(y_batch, A_conj):
            AHy = tf.einsum('ktrxyz,Nktr->Nxyz', A_conj, y_batch)
            mag = tf.abs(AHy)
            max_mag = tf.reduce_max(mag, axis=(1, 2, 3), keepdims=True)
            mag_norm = mag / (max_mag + 1e-12)
            return dnn(mag_norm[..., None], training=False)

    elif model_name == 'CV-Deep2S':
        dnn = src.get_CV_UNet(input_shape=(X, Y, Z, 2))
        dnn.load_weights("models_weights/experimental/CV-Deep2S/model_Nf15_SNR30_CV_exp.h5")

        def forward(y_batch, A_conj):
            AHy = tf.einsum('ktrxyz,Nktr->Nxyz', A_conj, y_batch)
            max_mag = tf.reduce_max(tf.abs(AHy), axis=(1, 2, 3), keepdims=True)
            AHy_norm = AHy / tf.cast(max_mag + 1e-12, tf.complex64)
            AHy_ri = tf.stack([tf.math.real(AHy_norm), tf.math.imag(AHy_norm)], axis=-1)
            return dnn(AHy_ri, training=False)

    elif model_name == 'Deep2SP+':
        dnn = tf.keras.models.load_model(
            "models_weights/experimental/Deep2S-Plus/model_Nf15_SNR30_Deep2SP_exp",
            compile=False
        )

        def forward(y_batch, A_conj=None):
            y_ri = tf.stack([tf.math.real(y_batch), tf.math.imag(y_batch)], axis=-1)
            return dnn(y_ri, training=False)

    else:
        raise ValueError(f"Unknown MODEL_TYPE: {model_name}")

    return dnn, forward


# ============================================================
# Target generation
# ============================================================
def generate_target_img(
    clean_img,
    victim_forward,
    sarRawData,                 # victim raw: (1,K,T,R)
    A_conj_tf,
    attack_mode="Full",         # "Full" or "ROI"
    target_mode="noise",        # "noise" or "object"
    seed=42,
    targetRawData=None,         # (1,K,T,R), only for object mode
):
    c = clean_img[..., 0] if len(clean_img.shape) == 5 else clean_img   # (1,X,Y,Z)
    Zc = int(c.shape[3])

    # ----------------------------
    # Build target
    # ----------------------------
    if target_mode.lower() == "noise":
        # shuffle victim raw
        y = sarRawData[0].numpy()               # (K,T,R)
        K, T, R = y.shape
        Np = T * R

        Xv = y.reshape(K, Np)
        rng = np.random.default_rng(seed)
        for k in range(K):
            Xv[k, :] = Xv[k, rng.permutation(Np)]

        y_shuf = tf.constant(Xv.reshape(K, T, R)[None, ...], dtype=tf.complex64)
        target_img = victim_forward(y_shuf, A_conj_tf)
        t = target_img[..., 0] if len(target_img.shape) == 5 else target_img

    elif target_mode.lower() == "object":
        if targetRawData is None:
            raise ValueError("For target_mode='object', pass targetRawData from the same 10-file loader.")

        target_img = victim_forward(targetRawData, A_conj_tf)
        t = target_img[..., 0] if len(target_img.shape) == 5 else target_img

        # optional scale match
        clean_mip = tf.reduce_max(c[0], axis=2).numpy().astype(np.float32)
        targ_mip  = tf.reduce_max(t[0], axis=2).numpy().astype(np.float32)

        s_clean = np.quantile(np.abs(clean_mip).reshape(-1), 0.995) + 1e-12
        s_targ  = np.quantile(np.abs(targ_mip).reshape(-1), 0.995) + 1e-12

        t = tf.cast(t, tf.float32) * tf.constant((s_clean / s_targ), tf.float32)

    else:
        raise ValueError("target_mode must be 'noise' or 'object'")

    # ----------------------------
    # ROI mask from CLEAN image
    # ----------------------------
    roi_mask = None
    if attack_mode == "ROI":
        c_np = c.numpy()[0]                     # (X,Y,Z)
        clean_mip = c_np.max(axis=2)

        thresh = 0.7 * clean_mip.max()
        binary = clean_mip > thresh

        labels, num_features = label(binary)
        if num_features > 0:
            largest_label = np.argmax(np.bincount(labels.ravel())[1:]) + 1
            roi_2d = binary_dilation(labels == largest_label, iterations=1)
            roi_3d = np.repeat(roi_2d[..., None], Zc, axis=2)
            roi_mask = tf.constant(roi_3d, dtype=tf.bool)

    return t, roi_mask


# ============================================================
# Helper: reconstruct attacked image from random A
# ============================================================
def reconstruct_adv_image_random_tf(X_v, D, A_rand, A_conj_tf, global_scale):
    """
    X_v    : (K,TR) complex
    D      : (K,TR) complex
    A_rand : (TR,) complex
    """
    A_row = tf.reshape(A_rand, (1, T * R))           # (1,TR)
    delta = D * A_row                                # (K,TR)
    Y_att = tf.reshape(X_v + delta, (1, K, T, R))    # (1,K,T,R)

    adv_img = victim_forward(Y_att, A_conj_tf)
    adv_img_s = adv_img / global_scale

    Pa = float(tf.math.real(tf.linalg.norm(delta) ** 2).numpy()) + EPS
    Pr = float(tf.math.real(tf.linalg.norm(X_v) ** 2).numpy()) + EPS
    PaPr = Pa / Pr
    PaPr_dB = 10.0 * math.log10(PaPr + EPS)

    return adv_img_s, PaPr, PaPr_dB


# ============================================================
# Helper: compare attacked image to clean image
# Uses MIP, consistent with your TF evaluation
# ============================================================
def compare_to_clean_tf(adv_img_s, clean_img_s):
    adv4 = adv_img_s[..., 0] if len(adv_img_s.shape) == 5 else adv_img_s
    cln4 = clean_img_s[..., 0] if len(clean_img_s.shape) == 5 else clean_img_s

    adv2d = tf.reduce_max(adv4[0], axis=2).numpy().astype(np.float32)
    cln2d = tf.reduce_max(cln4[0], axis=2).numpy().astype(np.float32)

    mse_AC = float(np.mean((adv2d - cln2d) ** 2))

    a = adv2d.reshape(-1)
    c = cln2d.reshape(-1)
    ncc_AC = float(np.sum(a * c) / (np.sqrt(np.sum(a**2) * np.sum(c**2)) + EPS))

    data_range_C = float(cln2d.max() - cln2d.min()) + EPS
    ssim_AC = float(ssim(adv2d, cln2d, data_range=data_range_C))
    psnr_AC = float(psnr(cln2d, adv2d, data_range=data_range_C))

    return mse_AC, ncc_AC, ssim_AC, psnr_AC




## Random W Attack

In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODEL_TYPE        = "CV-Deep2S"   # 'Deep2S', 'CV-Deep2S', 'Deep2SP+'
ATTACK_MODE       = "Full"        # 'Full' or 'ROI'
TARGET_MODE       = "noise"      # 'noise' or 'object'

EPS = 1e-12
ORIGIN = "upper"

VICTIM_RAW_SELECT = 10             # 1..10

TARGET_RAW_SELECT = 1             # only used if TARGET_MODE == "object"
DATA_DIR          = "data"
AUG_PREFIX        = "aug"

# random-weight settings
num_trials = 20
candidate_PaPr_dB = [10]#list(range(0, 31))   # 0:30

success_ssim_thresh = 0.001

success_rate_thresh = 0.01

random_weight_mode = "complex_gaussian"   # "complex_gaussian" or "unit_modulus"

tf.random.set_seed(0)
np.random.seed(0)

# ============================================================
# Load A operator + infer dimensions
# ============================================================
A = np.load("models_weights/experimental/A15_exp.npy")
K, T, R, X, Y, Z = A.shape
A_conj_tf = tf.math.conj(tf.constant(A.astype(np.complex64)))

# ============================================================
# Load victim raw
# ============================================================
sarRawData, sarRawData_temp, victim_path = load_raw_by_select(
    raw_select=VICTIM_RAW_SELECT,
    data_dir=DATA_DIR,
    aug_prefix=AUG_PREFIX
)
print(f"[VICTIM] {victim_path}")

# D is (K,TR) complex64, X_v is (K,TR) complex64
D   = tf.constant(np.load(os.path.join(DATA_DIR, "D_flat.npy")).astype(np.complex64))
X_v = tf.constant(sarRawData_temp.reshape(K, T * R), dtype=tf.complex64)

# ============================================================
# Build selected model pipeline
# ============================================================
dnn_model, victim_forward = get_model_pipeline(MODEL_TYPE, X, Y, Z)

# ============================================================
# Clean image
# ============================================================
clean_img = victim_forward(sarRawData, A_conj_tf)
c = clean_img[..., 0] if len(clean_img.shape) == 5 else clean_img
global_scale = tf.reduce_max(tf.abs(c)) + tf.constant(EPS, tf.float32)

# ============================================================
# Target raw (only if object mode)
# ============================================================
targetRawData = None
target_path = None
if TARGET_MODE.lower() == "object":
    targetRawData, _, target_path = load_raw_by_select(
        raw_select=TARGET_RAW_SELECT,
        data_dir=DATA_DIR,
        aug_prefix=AUG_PREFIX
    )
    print(f"[TARGET] {target_path}")

# ============================================================
# Target image + ROI mask
# ============================================================
target_img, roi_mask = generate_target_img(
    clean_img=clean_img,
    victim_forward=victim_forward,
    sarRawData=sarRawData,
    targetRawData=targetRawData,
    A_conj_tf=A_conj_tf,
    attack_mode=ATTACK_MODE,
    target_mode=TARGET_MODE,
    seed=42,
)

if (len(clean_img.shape) == 5) and (len(target_img.shape) == 4):
    target_img = target_img[..., None]

# shared scaling
clean_img_s  = clean_img / global_scale
#target_img_s = target_img / global_scale


# ============================================================
# RANDOM-WEIGHT ATTACK SWEEP
# ============================================================
Pr_ref = float(tf.math.real(tf.linalg.norm(X_v) ** 2).numpy()) + EPS

print("\n============================================================")
print(f"Random-weight concealment sweep for {MODEL_TYPE} on victim={VICTIM_RAW_SELECT}")
print(f"Trials per budget: {num_trials}")
print(f"Success criterion: SSIM(A,C) <= {success_ssim_thresh:.3f}")
print(f"Required success rate: {success_rate_thresh:.2f}")
print("============================================================\n")

all_budget_results = []
best_budget_found = False
chosen_budget_idx = None

for b_idx, target_PaPr_dB in enumerate(candidate_PaPr_dB):
    target_PaPr = 10 ** (target_PaPr_dB / 10.0)

    psnr_AC_trials = np.zeros(num_trials, dtype=np.float64)
    ssim_AC_trials = np.zeros(num_trials, dtype=np.float64)
    ncc_AC_trials  = np.zeros(num_trials, dtype=np.float64)
    pa_pr_trials   = np.zeros(num_trials, dtype=np.float64)
    mse_AC_trials  = np.zeros(num_trials, dtype=np.float64)

    success_flags = np.zeros(num_trials, dtype=bool)

    adv_imgs_trial = []
    A_store_trial  = []

    print(f"Testing budget Pa/Pr = {target_PaPr_dB:.2f} dB ...")

    for tr in range(num_trials):
        rng_local = np.random.default_rng(10000 + 100 * (b_idx + 1) + (tr + 1))

        if random_weight_mode.lower() == "complex_gaussian":
            A_re_np = rng_local.standard_normal(T * R).astype(np.float32)
            A_im_np = rng_local.standard_normal(T * R).astype(np.float32)
            A_rand_np = A_re_np + 1j * A_im_np

        elif random_weight_mode.lower() == "unit_modulus":
            theta = 2 * np.pi * rng_local.random(T * R).astype(np.float32)
            A_rand_np = np.exp(1j * theta).astype(np.complex64)

        else:
            raise ValueError("Unknown random_weight_mode.")

        A_rand = tf.constant(A_rand_np, dtype=tf.complex64)

        delta_rand = D * tf.reshape(A_rand, (1, T * R))
        Pa_rand = float(tf.math.real(tf.linalg.norm(delta_rand) ** 2).numpy()) + EPS

        scale_pow = math.sqrt((target_PaPr * Pr_ref) / (Pa_rand + EPS))
        A_rand = A_rand * tf.cast(scale_pow, tf.complex64)

        adv_img_s, PaPr_final, _ = reconstruct_adv_image_random_tf(
            X_v=X_v,
            D=D,
            A_rand=A_rand,
            A_conj_tf=A_conj_tf,
            global_scale=global_scale
        )

        mse_AC, ncc_AC, ssim_AC, psnr_AC = compare_to_clean_tf(adv_img_s, clean_img_s)

        mse_AC_trials[tr] = mse_AC
        ncc_AC_trials[tr] = ncc_AC
        ssim_AC_trials[tr] = ssim_AC
        psnr_AC_trials[tr] = psnr_AC
        pa_pr_trials[tr] = PaPr_final

        success_flags[tr] = (ssim_AC <= success_ssim_thresh)

        adv_imgs_trial.append(tf.identity(adv_img_s))
        A_store_trial.append(tf.identity(A_rand))

    success_rate = np.mean(success_flags)

    print(f"  PSNR(A,C): {psnr_AC_trials.mean():.2f} ± {psnr_AC_trials.std():.2f} dB")
    print(f"  SSIM(A,C): {ssim_AC_trials.mean():.4f} ± {ssim_AC_trials.std():.4f}")
    print(f"  NCC(A,C) : {ncc_AC_trials.mean():.4f} ± {ncc_AC_trials.std():.4f}")
    print(f"  Pa/Pr    : {pa_pr_trials.mean():.4e} ± {pa_pr_trials.std():.4e} (linear)")
    print(f"  Success rate (SSIM<={success_ssim_thresh:.3f}): {success_rate:.2f}\n")

    result = {
        "PaPr_dB": target_PaPr_dB,
        "PaPr_lin": float(pa_pr_trials.mean()),
        "psnr_AC_mean": float(psnr_AC_trials.mean()),
        "psnr_AC_std": float(psnr_AC_trials.std()),
        "ssim_AC_mean": float(ssim_AC_trials.mean()),
        "ssim_AC_std": float(ssim_AC_trials.std()),
        "ncc_AC_mean": float(ncc_AC_trials.mean()),
        "ncc_AC_std": float(ncc_AC_trials.std()),
        "mse_AC_mean": float(mse_AC_trials.mean()),
        "mse_AC_std": float(mse_AC_trials.std()),
        "success_rate": float(success_rate),
        "adv_imgs": adv_imgs_trial,
        "A_store": A_store_trial,
        "success_flags": success_flags.copy(),
        "psnr_AC_trials": psnr_AC_trials.copy(),
        "ssim_AC_trials": ssim_AC_trials.copy(),
        "ncc_AC_trials": ncc_AC_trials.copy(),
        "pa_pr_trials": pa_pr_trials.copy(),
    }
    all_budget_results.append(result)

    if (not best_budget_found) and (success_rate >= success_rate_thresh):
        best_budget_found = True
        chosen_budget_idx = b_idx

# ============================================================
# Choose operating budget
# ============================================================
if best_budget_found:
    print(f"Chosen smallest successful budget: {all_budget_results[chosen_budget_idx]['PaPr_dB']:.2f} dB")
else:
    chosen_budget_idx = len(all_budget_results) - 1
    print("No budget met success-rate criterion.")
    print(f"Using strongest tested budget: {all_budget_results[chosen_budget_idx]['PaPr_dB']:.2f} dB")

chosen = all_budget_results[chosen_budget_idx]

succ_idx = np.where(chosen["success_flags"])[0]
if len(succ_idx) > 0:
    ord_idx = np.argsort(chosen["ssim_AC_trials"][succ_idx])
    pick = succ_idx[ord_idx[max(0, round(len(ord_idx) / 2) - 1)]]
else:
    pick = int(np.argmin(chosen["ssim_AC_trials"]))

adv_img_best = chosen["adv_imgs"][pick]
A_best       = chosen["A_store"][pick]


## evaluation

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n============================================================")
print(f"FINAL TABLE NUMBERS for {MODEL_TYPE}")
print(f"Use budget: {chosen['PaPr_dB']:.2f} dB")
print(f"PSNR(A,C): {chosen['psnr_AC_mean']:.2f} ± {chosen['psnr_AC_std']:.2f} dB")
print(f"SSIM(A,C): {chosen['ssim_AC_mean']:.4f} ± {chosen['ssim_AC_std']:.4f}")
print(f"NCC(A,C) : {chosen['ncc_AC_mean']:.4f} ± {chosen['ncc_AC_std']:.4f}")

PaPr_lin_mean = chosen["PaPr_lin"]
PaPr_dB_mean  = 10.0 * math.log10(PaPr_lin_mean + EPS)
print(f"Pa/Pr    : {PaPr_lin_mean:.4f} (linear mean), {PaPr_dB_mean:.4f} dB")
print("============================================================")

# ============================================================
# BUDGET SWEEP FIGURE
# ============================================================
plt.figure()

xvals = [r["PaPr_dB"] for r in all_budget_results]
ssim_means = [r["ssim_AC_mean"] for r in all_budget_results]
succ_rates = [r["success_rate"] for r in all_budget_results]

ax1 = plt.gca()
ax1.plot(xvals, ssim_means, "-o", linewidth=1.5)
ax1.set_ylabel("Mean SSIM(A,C)")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(xvals, succ_rates, "-s", linewidth=1.5)
ax2.set_ylabel("Success rate")

ax1.set_xlabel("Pa/Pr (dB)")
plt.title(f"Random-weight concealment sweep: {MODEL_TYPE}")
plt.show()

# ============================================================
# QUALITATIVE FIGURE
# ============================================================
adv4 = adv_img_best[..., 0] if len(adv_img_best.shape) == 5 else adv_img_best
cln4 = clean_img_s[..., 0] if len(clean_img_s.shape) == 5 else clean_img_s

adv2d = tf.reduce_max(adv4[0], axis=2).numpy().astype(np.float32)
cln2d = tf.reduce_max(cln4[0], axis=2).numpy().astype(np.float32)

diff2d = adv2d - cln2d

plt.figure()

plt.subplot(1, 3, 1)
plt.imshow(cln2d, cmap="gray", origin=ORIGIN)
plt.title("Clean image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(adv2d, cmap="gray", origin=ORIGIN)
plt.title(f"Random attack image\nPa/Pr = {chosen['PaPr_dB']:.2f} dB")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(diff2d, cmap="gray", origin=ORIGIN)
plt.title("Difference (A-C)")
plt.axis("off")

plt.show()